- vllm 的两个场景
    - 推理/部署：offline batch generate、online deploy llm inference service
    - rl4llm training: rollout
        - vllm of verl: https://verl.readthedocs.io/en/latest/perf/perf_tuning.html#rollout-generation-tuning

- `vllm serve Qwen/Qwen2.5-1.5B-Instruct`

```sh
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --max-model-len 10240 \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen2.5-1.5b-instruct \
  --gpu-memory-utilization 0.85 \
  --api-key "abc123"
  
# dp=2
vllm serve $MODEL --data-parallel-size 2
```
- fastapi
    - `http://<vllm_host>:<port>/docs`
        - `http://127.0.0.1:8000/docs`
    - `http://<vllm_host>:<port>/metrics`: `http://127.0.0.1:8000/docs`
        - num_requests_waiting: 当前排队中的请求数
        - num_requests_running: 当前正在执行的请求数
- 分布式部署
    - https://docs.vllm.ai/en/stable/serving/data_parallel_deployment.html

### 参数

```
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --max-model-len 8192 \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen2.5-7b \
  --gpu-memory-utilization 0.85 \
  --enable-prefix-caching \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --api-key "abc123"
```

https://docs.vllm.ai/en/latest/configuration/optimization/
- `export VLLM_LOGGING_LEVEL=DEBUG`
    - 开启 debug 模式 (更详尽的日志输出)
        - vLLM API server version
    - 监控推理时的性能；
        - GPU KV cache usage
    - 线上部署时，结合日志具体调参分析
- --gpu-memory-utilization
    - 模型权重和 KV Cache (不包含 cuda graph)
- `--max-model-len` & `--max-num-seqs`:
    - max-model-len: 单条请求能用的「最大总 token 数（prompt + 输出）」
        - 默认是 llm 的 context window (`"max_position_embeddings": 32768,`)
    - --max-num-seqs：默认 256 ??
- --max-num-batched-tokens：每次前向里「总共喂进模型的 token 数上限（跨所有请求）」，单位是 token，不是 # reqs。
    - default：2048
    - Chunked Prefill：默认开启
    - chunked prefill = 用 max_num_batched_tokens 当「prefill 分块大小」来切 prompt。
- `--enforce-eager`: 要不要用 CUDA Graph。
    - 默认是 false，开启 cuda graph
        - Capturing CUDA graphs ... took 0.17 GiB
        - 这也是二者显存占用的差异点；
    - 开启 eager-mode（--enforce-eager）：完全关闭 CUDA Graph，只用普通 PyTorch eager 前向。
        - 启动也更快；
    - 关于 cuda graph
        - CUDA Graphs 需要额外的显存来“捕获”图。如果你的显存非常吃紧，甚至无法完成图的捕获（Graph Capture OOM），开启 Eager mode 可能是一个临时的救急方案。
- --enable-prefix-caching
    - 开启 Prefix Caching，服务器端的请求日志中，会有 prefix cache hit rate 相关的日志记录；
    - 跟 kv cache 的区别：
        - kvcache single request 内部自回归过程中的 kv cache 的缓存（用于 decoding generation），
        - prefix caching: 用于 Prefill (处理 Prompt) 阶段，跨请求共享；
- tool calling：工具调用和结构化输出 qwen2.5 以来都足够好；
```
# for qwen
--enable-auto-tool-choice \
--tool-call-parser hermes \
```

#### GPU block

> 操作系统的“页” (Page)，实现动态按需分配；

- 在传统的 LLM 推理中，显存分配通常是连续的。这就像在一个没有虚拟内存的旧电脑上，如果你需要存一个长文件，你必须在物理内存上找到一块完整的、连续的空地。如果找不到（即使零碎空间加起来够用），就会报错（OOM）。
- vLLM 的做法是：
    - 逻辑上连续，物理上离散： 就像 OS 把内存切成 4KB 的“页”一样，vLLM 把 KV Cache（键值缓存）切成固定大小的 Block。
    - Block Table（页表）： vLLM 维护一个映射表，记录“逻辑上的第几个 Block”对应“物理显存中的哪一块地址”。
- 一个 GPU Block 是 vLLM 显存管理的最小分配单位。
    - 它存储了固定数量 token 的 Key 和 Value 向量。
    - 在 vLLM 里，KV cache 不再是「一条序列一整块连续显存」，而是切成很多 固定大小的小块（block） 来管理，也就是 paged attention 的 “page”
- “GPU cache utilization” = 当前用了多少页
- 举一个例子（block_size = 4）
    - 用户输入 Prompt: "The sky is blue"。
    - 初始分配： vLLM 申请 物理 Block #7。
        - "The" -> 存入 Block #7, Slot 0
        - "sky" -> 存入 Block #7, Slot 1
        - "is" -> 存入 Block #7, Slot 2
        - "blue" -> 存入 Block #7, Slot 3
    - 动态生成： 模型生成下一个词 "and"。
        - 新分配： vLLM 发现 Block #7 已满，于是从显存池中抓取一个非连续的空闲块，比如 物理 Block #2。
        - "and" -> 存入 Block #2, Slot 0
    - 逻辑连接： 尽管物理上 Block #7 和 Block #2 可能相隔很远，但在计算 Attention 时，vLLM 的 CUDA Kernel 会通过 Block Table 知道它们逻辑上是连在一起的。

#### max-model-len

- 影响的是并发（物理的）
    - max-num-seqs：逻辑上的约束
    - 实际，两者取 min

```
# --max-model-len 32768 
Available KV cache memory: 5.29 GiB
GPU KV cache size: 98,960 tokens
Maximum concurrency for 32,768 tokens per request: 3.02x

# --max-model-len 8192
Available KV cache memory: 5.29 GiB
GPU KV cache size: 98,960 tokens
Maximum concurrency for 8,192 tokens per request: 12.08x
```

- max-num-batched-tokens：
    - max_concurrency * avg_tokens_per_seq

### kv cache

```
Available KV cache memory: 5.54 GiB
GPU KV cache size: 103,712 tokens
```
- $\text{Size}_{\text{token}} = 2 \times L \times N_{\text{kv}} \times D_{\text{head}} \times P_{\text{byte}}$
    - 以 qwen 2.5-7b 为例：https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/blob/main/config.json
    - 2\*28\*4\*128\*2=57344

In [1]:
2*28*4*128*2

57344

In [2]:
5.54 * 1024 * 1024 * 1024 / (2*28*4*128*2)

103734.12571428572

```
INFO ... num_gpu_blocks is: 6482
```
- block 对齐
    - 默认配置下，vLLM 的 Block Size = 16。
- $6482 \text{ blocks} \times 16 \text{ tokens/block} = \mathbf{103,712} \text{ tokens}$
    - Maximum concurrency for 8,192 tokens per request: 12.66x
    - 在假设每个请求都用满 8192 上下文长度的情况下，理论最大并发：103712/8192

### 显存占用分析

```
Free memory on device (23.2/23.65 GiB) on startup. Desired GPU memory utilization is (0.85, 20.1 GiB). Actual usage is 14.25 GiB for weight, 0.28 GiB for peak activation, 0.04 GiB for non-torch memory, and 0.17 GiB for CUDAGraph memory. Current kv cache memory in use is 5.54 GiB.
```

$$
\text{KV Cache} = (\text{显卡总显存} \times \text{利用率}) - \text{模型权重} - \text{运行时预留 (Activation Buffer)}
$$
- --gpu-memory-utilization 0.85：$\text{vLLM 可用总预算} = 23.65 \text{ GiB} \times 0.85 = \mathbf{20.1 \text{ GiB}}$
- 扣除“固定开销” (Model Weights)
    - (EngineCore_DP0 ...) INFO ... Model loading took 14.2488 GiB
    - $20.1 \text{ GiB} - 14.25 \text{ GiB} = \mathbf{5.85 \text{ GiB}}$
- 0.28 peak activation: 一次请求激活的探测（profile）
    - `Memory profiling takes 5.92 seconds"`
    - vLLM 在启动时用一批 dummy token（长度 = max_num_batched_tokens = 2048）跑了一次前向，把这次前向过程中 PyTorch 张量额外占用的峰值显存 记录下来。
    - During a profile run, peak activation memory is measured by forwarding a dummy input sequence with the maximum batch tokens (max_num_batched_tokens) through the model.
- https://medium.com/%40kimdoil1211/how-much-gpu-memory-do-you-really-need-for-efficient-llm-serving-4d26d5b8b95b

### chunked prefill & max_num_batched_tokens

- max_num_batched_tokens = 一次 GPU 前向里最多塞多少 token（所有请求加起来）
- chunked_prefill = 当某个 request 的 prompt 太长，塞不进这次前向，就把它切成几块来“分批灌入 KV cache”
    - 避免一个几万 token 的 prompt 一次占满整个 batch，拖垮其它请求的 ITL
```
GPU KV cache size: 103,712 tokens
Maximum concurrency for 8,192 tokens per request: 12.66x
```
- KV cache 决定了：最多能在显存里同时“挂着”多少 token 的上下文（103,712）
- max_num_batched_tokens 决定了：每一步前向最多处理多少 token（2048）
- chunked prefill 决定了：当某个 request 的 prompt 太长时，如何拆成多个“2048 以内的小段”去填满这个 103,712 token 的 KV 容量
- verl
    - Adjust max_num_seqs or max_num_batched_tokens. If the **GPU cache utilization** is relatively low in the log, increase max_num_seqs or max_num_batched_tokens can enlarge the effective batch size in the decoding stage, allowing more concurrent requests per batch. We recommend setting max_num_batched_tokens > 2048 for higher throughput.

### 两阶段的理解（prefill & decode）

- 在纯 decode 阶段，每条正在生成的序列每步只生成 1 个新 token，
    - tokens_in_batch ≈ num_decode_seqs
    - 如果 max_num_batched_tokens 远大于当前活跃 seq 数（比如 2048，而你就几十条并发），
        - → token 约束根本没打到上限，
        - → 这时增加 max_num_seqs 反而能多塞几条 seq 进来，整体 decode throughput 会提高（更多 token 并行）。
- Prefill + chunked prefill 阶段
    - 有 chunked prefill 时，长 prompt 会被切成块，调度器在一个 step 里可以：
        - 对一些 seq 做 prefill chunk（每条好几十/几百 token），不存在 `chunk size`这个参数
        - 对另一些 seq 做 decode 1 token
    - 当前可用的 Prefill 额度 = 8192 - (当前正在 Decode 的请求数量）
        - 假设你现在有 100 个并发请求正在生成（Decode 阶段），它们占用了 100 个 Token 的配额。那么系统还剩下 8192 - 100 = 8092 个 Token 的空间。调度器就会把排队中那个超长 Prompt（比如 20k 长度）切出 8092 个 Token 放入当前 Batch 进行 Prefill。
    - `total_tokens = sum(prefill_chunk_tokens) + num_decode_seqs`
    - `必须 ≤ max_num_batched_tokens`

> Chunked prefill is enabled with max_num_batched_tokens=8192

- 场景 A：只有一条长 prompt，没有其它 decode
    - max_num_batched_tokens = 8192
    - 当前没有 decode：decode_tokens = 0
    - prefill_budget = 8192
如果这条 prompt 长 10k token：
- 第 1 步：chunk1 大小 = min(10000, 8192) = 8192
    - 还剩 1808 token
- 第 2 步：chunk2 大小 = min(1808, 8192) = 1808
    - prefill 完成，后面开始正常 decode

从外部看，你就会看到这个长请求被切成 “8192 + 1808” 这两块。这就是我之前说的“每次切一大 chunk”。

------

场景 B：有很多 decode，在 decode 阶段插进来一个新长 prompt

例如：

正在 decode 的 seq 有 N_decode = 256，max_num_batched_tokens = 8192
- decode_tokens = 256
- prefill_budget = 8192 - 256 = 7936
  
如果这次只加了一条长 prompt：本步 prefill chunk 大小 = min(剩余 prompt 长度, 7936)

### 单机多卡部署多个服务

```
# 服务 A
CUDA_VISIBLE_DEVICES=0,1,2,3 \
MASTER_ADDR=127.0.0.1 MASTER_PORT=29581 \
nohup vllm serve Qwen/Qwen2.5-7B-Instruct \
  --data-parallel-size 4 \
  --max-model-len 10240 \
  --host 0.0.0.0 --port 8000 \
  --api-key "abc123" \
  --served-model-name qwen7b-a > vllmA.log 2>&1 &

# 服务 B
CUDA_VISIBLE_DEVICES=4,5,6,7 \
MASTER_ADDR=127.0.0.1 MASTER_PORT=29582 \
nohup vllm serve Qwen/Qwen2.5-7B-Instruct \
  --data-parallel-size 4 \
  --max-model-len 10240 \
  --host 0.0.0.0 --port 8001 \
  --api-key "abc123" \
  --served-model-name qwen7b-b > vllmB.log 2>&1 &
```